# Code to calculate the threshold data for the OA extremes calculation

based on the 90% value compute from daily data for 20 years

1. need to use the megamem (megamembw works too) queue (python3 !! analysis3)
2. struggled to get papermill to work
3. could further adjust chunking to speed up execution
4. could explore use of histogram for the calculation



In [ ]:
import xarray as xr
from dask.distributed import Client
import dask.array as da
from dask import delayed
import numpy as np
import os
# importing sys
import sys

In [ ]:
client = Client()
print("Dask dashboard available at:", client.dashboard_link)
client

In [ ]:
# parameters
oax_var='OAR'
oax_var='PH'

In [ ]:
coords_to_drop =['ST_OCEAN']
vars_to_drop =['XT_OCEAN_bnds','XT_OCEAN_bnds']

# preprocesser to drop unwanted variables
def drop_stuff(ds, coords_to_drop,vars_to_drop):
    """
    Preprocessor function to drop specified coordinates and variables from a dataset loaded via xr.open_mfdataset

    Parameters:
        ds (xarray.Dataset): The dataset from which coordinates & variables are to be dropped.
        coords_to_drop (list of str): List of coordinate names to drop.
        vars_to_drop(list of str): List of variable names to drop

    Returns:
        xarray.Dataset: Dataset with specified coordinates and variables dropped.
    """
    # Drop coordinates if they are in the dataset
    ds = ds.drop_vars(coords_to_drop, errors='ignore')
    ds = ds.drop_vars(vars_to_drop, errors='ignore')
    return ds

In [ ]:
def process_threshold1(ds, time_dim, start, end, variable, period):
    # Rechunk the data along the time dimension
    data = getattr(ds, variable).sel(**{time_dim: slice(start, end)}).chunk({time_dim : -1, 
                                'xt_ocean': 120, 'yt_ocean': 60}).persist()
#    data = getattr(ds, variable).sel(**{time_dim: slice(start, end)}).persist()

    # Debug: Print the shape and chunking of the data
    print(f"Processing period {period}, data shape: {data.shape}, chunks: {data.chunks}")

    # Calculate the 90th percentile
    p90_month=0
#    p90_month = data.groupby('Time.month').quantile(0.9, dim='Time') # .compute()
    p90_day = data.groupby('Time.dayofyear').quantile(0.9, dim=time_dim) # .compute()
    dtmp=data.groupby('Time.dayofyear')
    print(dtmp)
#    p90_day = da.percentile(dtmp, q=[90], axis=0, method="tdigest" ).persist()
    return p90_day, p90_month



In [ ]:
%%time
# Define the periods for processing
from glob import glob
GWL_periods = { 'GW4p0': ('2074-01-01', '2093-12-31') }
GWL_periods = {
    'current': ('1995-01-01', '2014-12-31'),
    'GW1p2': ('2001-01-01', '2020-12-31'),
    'GW1p5': ('2015-01-01', '2034-12-31'),
    'GW2p0': ('2030-01-01', '2049-12-31'),
    'GW3p0': ('2053-01-01', '2072-12-31'),
    'GW4p0': ('2074-01-01', '2093-12-31')
}


# Directory paths for OAE
dir1_new = '/scratch/xv83/rxm599/historical/'
dir2_new = '/scratch/xv83/rxm599/future/'
# modified to get append historical data to start of future output
pattern0 = sorted(glob(dir1_new + 'files*.nc'))
pattern1 = sorted(glob(dir1_new + 'files201[4-9]*.nc'))
pattern2 = sorted(glob(dir2_new + 'files*.nc'))

# Load datasets with chunking
dsst1 = xr.open_mfdataset(pattern0, parallel=True,
                          preprocess=lambda x: drop_stuff(x, coords_to_drop, vars_to_drop)).squeeze()
dsst2 = xr.open_mfdataset(pattern1 + pattern2, parallel=True,
                          preprocess=lambda x: drop_stuff(x, coords_to_drop, vars_to_drop)).squeeze()
#parallel=True, chunks={'TIME41': 10})
#chunks={'Time': 10}

In [ ]:
#rename coordinates!!
dsst2 = dsst2.rename({"TIME11": "Time","XT_OCEAN": "xt_ocean", "YT_OCEAN": "yt_ocean" })
dsst1 = dsst1.rename({"TIME11": "Time","XT_OCEAN": "xt_ocean", "YT_OCEAN": "yt_ocean" })

In [ ]:
%%time
# Process oax_var
for period, (start, end) in GWL_periods.items():
    if period == 'current':
        p90d,p90m = process_threshold1(dsst1*(-1), 'Time', start, end, oax_var, period)
    else:
        p90d,p90m = process_threshold1(dsst2*(-1), 'Time', start, end, oax_var, period)
    file1 = f'/scratch/xv83/rxm599/{oax_var}_percentile_daily_{period}.nc'
    # Save 90th percentile as netCDF
    p90d.name='raw90'
    p90d.to_netcdf(file1, compute=True)

In [ ]:
%%time
for period, (start, end) in GWL_periods.items():
    print(period)
    file1 = f'/scratch/xv83/rxm599/{oax_var}_percentile_daily_{period}.nc'
    d1=xr.open_dataset(file1, chunks={'dayofyear': -1, 'yt_ocean': 300, 'xt_ocean': 600 } )
    p90d=d1.raw90
    print(p90d)
# smoothing with sm running mean
    sm=31
    pnew = xr.concat([p90d[-sm:],p90d,p90d[0:sm]],dim='dayofyear')
    pnew = pnew.assign_coords(dayofyear=np.arange(-sm,366+sm))
    a=(pnew.rolling(dayofyear=sm,center=True).mean())
    b=(a.sel(dayofyear=slice(1,366)) )
    file2 = f'/scratch/xv83/rxm599/{oax_var}_percentile_sdaily_{period}.nc'
    b.name='smooth90'
    b.to_netcdf(file2, compute=True)
    

In [ ]:
d1

In [ ]:
sys.exit(1)

In [1]:
# select a point to test the MHW code
xlat=10; xlon= 190
sst1 = sst.sel(yt_ocean=xlat, xt_ocean=xlon, method='nearest')
ssth1 = ssth.sel(yt_ocean=xlat, xt_ocean=xlon, method='nearest')

NameError: name 'sst' is not defined

In [ ]:
oax_var="OAR"
process_all_periods1(GWL_periods, sst, ssth, oax_var)

In [ ]:
%%time
# Process SST
for period, (start, end) in GWL_periods.items():
    if period == 'current':
        p90d,p90m = process_threshold1(dsst1, 'Time', start, end, 'temp', period)
        

In [ ]:
%%time
p90d.to_netcdf('/scratch/xv83/rxm599/p90d_all.nc',mode='w')
p90m.to_netcdf('/scratch/xv83/rxm599/p90m_all.nc',mode='w')

In [ ]:
p90d.sel(xt_ocean=150,yt_ocean=-40, method='nearest').to_netcdf('/scratch/xv83/rxm599/p90d.nc',mode='w')
p90m.sel(xt_ocean=150,yt_ocean=-40, method='nearest').to_netcdf('/scratch/xv83/rxm599/p90m.nc',mode='w')
p90d.sel(xt_ocean=150,yt_ocean=-40,method='nearest').plot() 

In [ ]:
pp90m=xr.open_dataset('/scratch/xv83/rxm599/p90m.nc')
pp90d=xr.open_dataset('/scratch/xv83/rxm599/p90d.nc')
pp90d.temp.plot()
pp90m.temp.plot()
p90m=pp90m.temp
p90d=pp90d.temp

In [ ]:
import pandas as pd
# Step 1: Create a daily time index for a non-leap year
time = pd.date_range("2000-01-01", "2000-12-31", freq="D")
# Step 2: Extract month numbers for each day
month_for_day = xr.DataArray(time.month, coords={"time": time}, dims="time")
# Step 3: Map daily months to monthly climatology
clim_daily = p90m.sel(month=month_for_day)

#clim_daily.isel(xt_ocean=150,yt_ocean=-40).plot() 

n_days = len(time)  # 365 or 366
#fractional_month = (np.arange(n_days) * 12 / n_days) + 1
fractional_month = np.linspace(1, 13, n_days, endpoint=False)
pext = xr.concat([p90m,p90m.sel(month=1)],dim='month')
pext = pext.assign_coords(month=np.arange(1, 14))
daily = pext.interp( month=xr.DataArray(fractional_month, coords={"time": time}, dims="time") )
# Step 3: Interpolate
daily = pext.interp(month=xr.DataArray(fractional_month, coords={"dayofyear": np.arange(1, n_days+1)}, dims="dayofyear") )


# Step 1: Define mid-month positions (0.5, 1.5, ..., 11.5, 12.5)
mid_months = np.arange(0.5, 12.5 + 1, 1)  # 0.5 .. 12.5
pext = pext.assign_coords(month=mid_months)

# Step 3: Fractional month coordinate for each day
fractional_month = np.linspace(0.5, 12.5, n_days)
# Step 4: Interpolate to daily values
#daily = pext.interp(month=xr.DataArray(fractional_month, coords={"dayofyear": np.arange(1, n_days+1)}, dims="dayofyear") )

daily.plot() 
p90d.plot() 

In [ ]:
pext = xr.concat([p90m,p90m.sel(month=1)],dim='month')
# Step 1: Define mid-month positions (0.5, 1.5, ..., 11.5, 12.5)
mid_months = np.arange(0.5, 12.5 + 1, 1)  # 0.5 .. 12.5
pext = pext.assign_coords(month=mid_months)
# Step 3: Fractional month coordinate for each day
fractional_month = np.linspace(0.5, 12.5, n_days)
# Step 4: Interpolate to daily values
daily = pext.interp(month=xr.DataArray(fractional_month, coords={"dayofyear": np.arange(1, n_days+1)}, dims="dayofyear") )

daily.plot() 
p90d.plot() 

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

def daily_climatology_from_monthly(clim_monthly: xr.DataArray, n_days: int = 365) -> xr.DataArray:
    """
    Smooth daily climatology from monthly means using mid-month DOY positions
    with proper wrap-around (Dec->Jan). Output is indexed by dayofyear (1..n_days).

    Parameters
    ----------
    clim_monthly : xr.DataArray
        Monthly climatology with dim 'month' taking values 1..12.
        Works with extra dims (e.g., lat/lon).
    n_days : int
        365 or 366.

    Returns
    -------
    xr.DataArray
        Daily climatology on 'dayofyear' (1..n_days).
    """
    if "month" not in clim_monthly.dims:
        raise ValueError("clim_monthly must have a 'month' dimension (1..12).")

    # Choose a representative year with the correct day count
    year = 2001 if n_days == 365 else 2000  # 2000 is leap year

    # Mid-month DOY centers (accounting for real month lengths)
    centers = []
    for m in range(1, 13):
        start = pd.Timestamp(year, m, 1)
        next_start = pd.Timestamp(year + (m // 12), (m % 12) + 1, 1)
        L = (next_start - start).days  # days in month
        mid = start.dayofyear + (L - 1) / 2.0
        centers.append(mid)
    centers = np.array(centers, dtype=float)

    # Prepend Dec (previous year) and append Jan (next year) for periodic wrap
    centers_ext = np.concatenate(([centers[-1] - n_days], centers, [centers[0] + n_days]))

    # Extend the data along 'month' then attach DOY centers and rename the axis
    clim_ext = xr.concat(
        [clim_monthly.sel(month=12), clim_monthly, clim_monthly.sel(month=1)],
        dim="month"
    ).rename(month="doy_center").assign_coords(doy_center=("doy_center", centers_ext))

    # Target DOY grid
    doy = np.arange(1, n_days + 1)

    # Interpolate in DOY space; output dimension becomes 'dayofyear'
    daily = clim_ext.interp(
        doy_center=xr.DataArray(doy, dims="dayofyear", coords={"dayofyear": doy})
    )

    return daily

daily_366 = daily_climatology_from_monthly(p90m, n_days=366)

daily_366.plot()
p90d.plot()

# smoothing with sm running mean
sm=31
pnew = xr.concat([p90d[-sm:],p90d,p90d[0:sm]],dim='dayofyear')
pnew = pnew.assign_coords(dayofyear=np.arange(-sm,366+sm))
a=(pnew.rolling(dayofyear=sm,center=True).mean())

(a.sel(dayofyear=slice(1,366)) ).plot()

In [ ]:
b=(a.sel(dayofyear=slice(1,366)) )
b

In [ ]:
#client.close()